# History feature engineering

Two-stage experiment for **history features** used in [`3 fatigue_modeling.ipynb`](3%20fatigue_modeling.ipynb):

1. **Tune construction** — Optuna over `ewma_alpha` and `rolling_window` (same 7 history columns).
2. **Ablation** — with best individual history feature construction fixed, find which history columns to keep (leave-one-out ranking + forward selection).

Decisions use **GroupKFold CV on train/val only**. Held-out test is used once in §3.

**Proxy model:** `catboost_ordinal` with fixed hyperparameters (`HISTORY_PROXY_PARAMS` in src/modeling/config.py — Optuna best from `catboost_ordinal_history` in the main notebook §3 History). There is **no model Optuna** in this notebook; only history construction (§1) and feature subset (§2) vary -- only tune one thing at a time.

These CatBoost params were tuned for one history setup in [`3 fatigue_modeling.ipynb`](3%20fatigue_modeling.ipynb) (default `EWMA_ALPHA` / `ROLLING_WINDOW`, all 7 features). When ablation changes columns or construction, they may no longer be globally optimal — but they are a fixed, realistic measuring stick. After applying feature engineering recommendations, re-run §3 History in the main notebook for fresh full model Optuna.

In [ ]:
%pip install -q -r ../../requirements.txt

In [ ]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.config import (
    DATA_PATH,
    EWMA_ALPHA,
    HISTORY_ABLATION_MODEL,
    HISTORY_FEATURES,
    HISTORY_PROXY_PARAMS,
    HISTORY_TUNING_TRIALS,
    N_CV_FOLDS,
    ROLLING_WINDOW,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.history_tuning import (
    cv_mae_with_history,
    prepare_tuned_bundle,
    run_forward_selection,
    run_leave_one_out_ablation,
    summarize_history_recommendation,
    test_mae_with_history,
    tune_history_construction,
)


## Load data and split


In [ ]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle))
print('Proxy model:', HISTORY_ABLATION_MODEL)
print('Proxy params:', HISTORY_PROXY_PARAMS)
print('Default construction:', f'ewma_alpha={EWMA_ALPHA}', f'rolling_window={ROLLING_WINDOW}')


## 1. Tune history construction

Optuna search over `ewma_alpha` ∈ [0.1, 0.5] and `rolling_window` ∈ {2, 3, 5, 7}. All 7 history columns are included. The proxy model is `catboost_ordinal` with fixed `HISTORY_PROXY_PARAMS` (no model Optuna here).


In [ ]:
default_cv_mae = cv_mae_with_history(
    df,
    bundle.train_val_mask,
    bundle.test_mask,
    bundle.y_ord_train_val,
    bundle.groups_train_val,
    model_name=HISTORY_ABLATION_MODEL,
    ewma_alpha=EWMA_ALPHA,
    rolling_window=ROLLING_WINDOW,
    history_cols=list(HISTORY_FEATURES),
    n_splits=N_CV_FOLDS,
    test_ids=bundle.test_ids,
)
print(f'Default construction CV MAE: {default_cv_mae:.4f}')

construction_result = tune_history_construction(
    df,
    bundle,
    model_name=HISTORY_ABLATION_MODEL,
    n_trials=HISTORY_TUNING_TRIALS,
    n_splits=N_CV_FOLDS,
)
best_alpha = construction_result['best_params']['ewma_alpha']
best_window = int(construction_result['best_params']['rolling_window'])
print(f"Best construction: ewma_alpha={best_alpha:.4f}, rolling_window={best_window}")
print(f"Tuned CV MAE: {construction_result['best_cv_mae']:.4f}")
print(f"Delta vs default: {construction_result['best_cv_mae'] - default_cv_mae:+.4f}")


## 2. History feature ablation

Rebuild the bundle with tuned construction params, then:
- **Leave-one-out:** drop each history column; higher `cv_mae_increase_vs_all` = more important.
- **Forward selection:** start from base features only; greedily add columns while CV MAE improves.

Same proxy model and fixed `HISTORY_PROXY_PARAMS` as §1.

Rolling column names (e.g. `activity_logsum_roll3_mean`) are unchanged when `rolling_window != 3`.


In [ ]:
tuned_bundle = prepare_tuned_bundle(df, best_alpha, best_window)

all_features_cv_mae, feature_importance = run_leave_one_out_ablation(
    df,
    tuned_bundle,
    ewma_alpha=best_alpha,
    rolling_window=best_window,
    model_name=HISTORY_ABLATION_MODEL,
    n_splits=N_CV_FOLDS,
)
print(f'All-{len(HISTORY_FEATURES)}-feature CV MAE: {all_features_cv_mae:.4f}')
print('Removing each history feature — higher cv_mae_increase_vs_all = more important:')
display(feature_importance)


In [ ]:
forward_selected, forward_cv_mae, forward_path = run_forward_selection(
    df,
    tuned_bundle,
    ewma_alpha=best_alpha,
    rolling_window=best_window,
    model_name=HISTORY_ABLATION_MODEL,
    n_splits=N_CV_FOLDS,
)
print('Forward selection path:')
display(forward_path)
print(f'Recommended subset ({len(forward_selected)} features): {forward_selected}')
print(f'Forward-selection CV MAE: {forward_cv_mae:.4f}')

recommendation = summarize_history_recommendation(
    construction_result,
    forward_selected,
    forward_cv_mae,
    default_cv_mae=default_cv_mae,
    feature_importance=feature_importance,
)
pd.Series(recommendation)


## 3. One-shot test evaluation

Single held-out test MAE with the recommended construction params and forward-selected history subset. Same `catboost_ordinal` proxy and `HISTORY_PROXY_PARAMS`. Use for reporting only — not for tuning decisions.


In [ ]:
test_mae = test_mae_with_history(
    df,
    tuned_bundle.train_val_mask,
    tuned_bundle.test_mask,
    tuned_bundle.y_ord_train_val,
    tuned_bundle.y_ord_test,
    model_name=HISTORY_ABLATION_MODEL,
    ewma_alpha=best_alpha,
    rolling_window=best_window,
    history_cols=forward_selected,
)
print(f'Proxy test MAE (recommended config): {test_mae:.4f}')


## 4. Apply to main pipeline

When satisfied with the results above, copy into [`src/modeling/config.py`](../../src/modeling/config.py):

```python
EWMA_ALPHA = ...        # from recommendation['ewma_alpha']
ROLLING_WINDOW = ...    # from recommendation['rolling_window']
HISTORY_FEATURES = [    # from recommendation['history_features']
    ...,
]
```

Then re-run §3 History in [`3 fatigue_modeling.ipynb`](3%20fatigue_modeling.ipynb) (full model Optuna tuning, not just the proxy).


In [ ]:
print('Suggested config.py updates:')
print(f'EWMA_ALPHA = {recommendation["ewma_alpha"]:.6f}')
print(f'ROLLING_WINDOW = {recommendation["rolling_window"]}')
print('HISTORY_FEATURES = [')
for feature in recommendation['history_features']:
    print(f'    "{feature}",')
print(']')
